# Layer 3 — 慢機台下鑽

按 device_id 聚合 device_duration，找出慢機台。
使用 gap detection 自動識別慢機台群組（而非固定 top-N）。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

sys_flags = pd.read_csv('../data/system_anomaly_flags.csv')
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')
df = df.merge(sys_flags, on='order_id')
df = df.merge(usr_flags, on='order_id')

normal = df[~(df['is_system_anomaly'] | df['is_user_anomaly'])].copy()
print(f"Normal orders: {len(normal)}")


Normal orders: 28483


## 1. Device Performance Ranking + Gap Detection

In [2]:
# Aggregate device-level stats
device_perf = normal.groupby('device_id').agg(
    device_dur_median=('device_duration_avg_seconds', 'median'),
    device_dur_mean=('device_duration_avg_seconds', 'mean'),
    device_dur_p95=('device_duration_avg_seconds', lambda x: x.quantile(0.95)),
    order_count=('order_id', 'count'),
    avg_file_count=('file_count', 'mean'),
    total_dur_median=('total_duration_seconds', 'median'),
).reset_index()

# Merge location info
loc_info = df.groupby('device_id').agg(
    loc_1=('loc_1', 'first'),
    loc_2=('loc_2', 'first'),
    system_name=('system_name', 'first'),
    device_mode_name=('device_mode_name', 'first'),
).reset_index()
device_perf = device_perf.merge(loc_info, on='device_id')

# Gap detection: sort by median desc, find largest gap
sorted_perf = device_perf.sort_values('device_dur_median', ascending=False).reset_index(drop=True)
sorted_perf['gap_to_next'] = sorted_perf['device_dur_median'].diff(-1).abs()

# Find the largest gap
max_gap_idx = sorted_perf['gap_to_next'].idxmax()
gap_value = sorted_perf.loc[max_gap_idx, 'gap_to_next']
cutoff = sorted_perf.loc[max_gap_idx, 'device_dur_median']

# Slow devices: those above the gap
slow_devices = sorted_perf.loc[:max_gap_idx]
n_slow = len(slow_devices)

print(f"Gap detection: largest gap = {gap_value:.1f}s at position {max_gap_idx}")
print(f"Cutoff: devices with median > {cutoff - gap_value:.1f}s are slow")
print(f"Slow devices identified: {n_slow}")
print(f"\nSlow Devices:")
print(slow_devices[['device_id', 'device_dur_median', 'device_dur_mean', 'device_dur_p95',
                     'order_count', 'loc_1', 'system_name']].to_string(index=False))


Gap detection: largest gap = 9.0s at position 7
Cutoff: devices with median > 3.0s are slow
Slow devices identified: 8

Slow Devices:
device_id  device_dur_median  device_dur_mean  device_dur_p95  order_count loc_1 system_name
 DEV-0062               14.0        13.809859           21.00          142 FAB-B   SYS-ALPHA
 DEV-0163               13.0        13.152439           20.85          164 FAB-B    SYS-BETA
 DEV-0035               13.0        13.241379           20.00          145 FAB-C   SYS-GAMMA
 DEV-0006               13.0        13.380282           21.00          142 FAB-A    SYS-BETA
 DEV-0189               13.0        13.215278           21.00          144 FAB-A    SYS-BETA
 DEV-0057               13.0        13.746575           20.75          146 FAB-A   SYS-GAMMA
 DEV-0028               13.0        13.437037           21.00          135 FAB-C   SYS-GAMMA
 DEV-0070               12.0        13.066116           21.00          121 FAB-C   SYS-GAMMA


## 2. Ground Truth 驗證

In [3]:
# Validate against ground truth
labels = pd.read_csv('../data/orders_with_labels.csv')
df_val = df.merge(labels[['order_id', '_is_slow_device']], on='order_id')
true_slow_devices = set(df_val[df_val['_is_slow_device']]['device_id'].unique())
pred_slow_devices = set(slow_devices['device_id'])

print(f"Ground truth slow devices: {len(true_slow_devices)}")
print(f"Predicted slow devices: {len(pred_slow_devices)}")
print(f"Intersection: {len(true_slow_devices & pred_slow_devices)}")
print(f"False positives: {pred_slow_devices - true_slow_devices}")
print(f"False negatives: {true_slow_devices - pred_slow_devices}")
print(f"Match: {'PERFECT' if pred_slow_devices == true_slow_devices else 'MISMATCH'}")


Ground truth slow devices: 8
Predicted slow devices: 8
Intersection: 8
False positives: set()
False negatives: set()
Match: PERFECT


## 3. 圖表

In [4]:
# Chart 1: Device ranking with gap detection highlight
top30 = device_perf.nlargest(30, 'device_dur_median')
slow_ids = set(slow_devices['device_id'])

fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#e74c3c' if d in slow_ids else '#3498db' for d in top30['device_id']]
ax.barh(range(len(top30)), top30['device_dur_median'].values, color=colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30['device_id'].values, fontsize=8)
ax.set_title(f'Device Performance Ranking (Red = {n_slow} Slow Devices by Gap Detection)')
ax.set_xlabel('Median Device Duration Avg (seconds)')
ax.invert_yaxis()

# Add horizontal line at gap boundary (between slow and normal groups)
ax.axhline(y=max_gap_idx + 0.5, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Gap boundary')

for i, (dur, cnt) in enumerate(zip(top30['device_dur_median'], top30['order_count'])):
    ax.text(dur + 0.3, i, f'n={cnt}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_device_ranking.png', dpi=150)
plt.close()
print("Saved: 3_device_ranking.png")


Saved: 3_device_ranking.png


In [5]:
# Chart 2: Faceted analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# By loc_1
loc1_stats = normal.groupby('loc_1')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
loc1_stats['median'].plot(kind='bar', ax=axes[0][0], color=sns.color_palette('Set2'))
axes[0][0].set_title('Median Device Duration by Location (loc_1)')
axes[0][0].set_ylabel('Median Duration (seconds)')
axes[0][0].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(loc1_stats.iterrows()):
    axes[0][0].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By system_name
sys_stats = normal.groupby('system_name')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
sys_stats['median'].plot(kind='bar', ax=axes[0][1], color=sns.color_palette('Set2'))
axes[0][1].set_title('Median Device Duration by System')
axes[0][1].set_ylabel('Median Duration (seconds)')
axes[0][1].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(sys_stats.iterrows()):
    axes[0][1].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By loc_2
loc2_data = normal.dropna(subset=['loc_2'])
loc2_stats = loc2_data.groupby('loc_2')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
loc2_stats['median'].plot(kind='bar', ax=axes[1][0], color=sns.color_palette('Set2'))
axes[1][0].set_title('Median Device Duration by Location (loc_2)')
axes[1][0].set_ylabel('Median Duration (seconds)')
axes[1][0].tick_params(axis='x', rotation=45)
for i, (idx, row) in enumerate(loc2_stats.iterrows()):
    axes[1][0].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By device_mode_name
mode_data = normal.dropna(subset=['device_mode_name'])
mode_stats = mode_data.groupby('device_mode_name')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
mode_stats['median'].plot(kind='bar', ax=axes[1][1], color=sns.color_palette('Set2'))
axes[1][1].set_title('Median Device Duration by Device Mode')
axes[1][1].set_ylabel('Median Duration (seconds)')
axes[1][1].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(mode_stats.iterrows()):
    axes[1][1].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_facet_analysis.png', dpi=150)
plt.close()
print("Saved: 3_facet_analysis.png")


Saved: 3_facet_analysis.png


In [6]:
# Chart 3: Slow device phase breakdown
slow_orders = normal[normal['device_id'].isin(slow_ids)].copy()

slow_orders['est_device'] = slow_orders['device_duration_avg_seconds'] * slow_orders['file_count'] / PARALLELISM
slow_orders['est_db'] = slow_orders['db_duration_avg_seconds'] * slow_orders['file_count'] / PARALLELISM
slow_orders['est_inner'] = slow_orders['inner_processing_duration_avg_seconds'] * slow_orders['file_count'] / PARALLELISM
slow_orders['est_queue'] = slow_orders['queue_duration_seconds']

fig, ax = plt.subplots(figsize=(14, 6))
phase_by_device = slow_orders.groupby('device_id')[['est_queue', 'est_db', 'est_device', 'est_inner']].mean()
phase_by_device.columns = ['Queue', 'DB', 'Device', 'Inner']
phase_by_device = phase_by_device.sort_values('Device', ascending=True)

phase_by_device.plot(kind='barh', stacked=True, ax=ax,
                     color=['#f39c12', '#3498db', '#e74c3c', '#2ecc71'])
ax.set_title(f'Phase Breakdown for {n_slow} Slow Devices')
ax.set_xlabel('Mean Estimated Duration (seconds)')
ax.set_ylabel('Device ID')
ax.legend(title='Phase')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_slow_device_breakdown.png', dpi=150)
plt.close()
print("Saved: 3_slow_device_breakdown.png")


Saved: 3_slow_device_breakdown.png


## 4. Summary

In [7]:
print(f"=== Layer 3 Summary ===")
print(f"Slow devices identified: {n_slow} (via gap detection, gap={gap_value:.1f}s)")
print(f"\nSlow Devices:")
for _, row in slow_devices.iterrows():
    print(f"  {row['device_id']}: median={row['device_dur_median']:.1f}s, "
          f"p95={row['device_dur_p95']:.1f}s, orders={row['order_count']}, "
          f"loc={row['loc_1']}/{row['loc_2']}, sys={row['system_name']}")

print(f"\nLocation breakdown:")
for loc, row in loc1_stats.iterrows():
    print(f"  {loc}: median={row['median']:.1f}s ({int(row['count'])} orders)")

# Data-driven facet conclusion
for facet_name, stats in [('loc_1', loc1_stats), ('system_name', sys_stats)]:
    ratio = stats['median'].max() / stats['median'].min() if stats['median'].min() > 0 else float('inf')
    if ratio > 1.5:
        print(f"\n⚠️  {facet_name} 有顯著差異 (max/min median ratio = {ratio:.1f}x)，值得進一步下鑽。")
    else:
        print(f"\n✓ {facet_name} 無顯著差異 (max/min median ratio = {ratio:.1f}x)，慢機台問題是 device-specific。")


=== Layer 3 Summary ===
Slow devices identified: 8 (via gap detection, gap=9.0s)

Slow Devices:
  DEV-0062: median=14.0s, p95=21.0s, orders=142, loc=FAB-B/None, sys=SYS-ALPHA
  DEV-0163: median=13.0s, p95=20.8s, orders=164, loc=FAB-B/None, sys=SYS-BETA
  DEV-0035: median=13.0s, p95=20.0s, orders=145, loc=FAB-C/AREA-1, sys=SYS-GAMMA
  DEV-0006: median=13.0s, p95=21.0s, orders=142, loc=FAB-A/AREA-2, sys=SYS-BETA
  DEV-0189: median=13.0s, p95=21.0s, orders=144, loc=FAB-A/AREA-2, sys=SYS-BETA
  DEV-0057: median=13.0s, p95=20.8s, orders=146, loc=FAB-A/AREA-2, sys=SYS-GAMMA
  DEV-0028: median=13.0s, p95=21.0s, orders=135, loc=FAB-C/None, sys=SYS-GAMMA
  DEV-0070: median=12.0s, p95=21.0s, orders=121, loc=FAB-C/AREA-1, sys=SYS-GAMMA

Location breakdown:
  FAB-A: median=3.0s (8130 orders)
  FAB-B: median=3.0s (9914 orders)
  FAB-C: median=3.0s (10439 orders)

✓ loc_1 無顯著差異 (max/min median ratio = 1.0x)，慢機台問題是 device-specific。

✓ system_name 無顯著差異 (max/min median ratio = 1.0x)，慢機台問題是 device-spec